In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from spx_history import *

- for one symbol first

In [3]:
start_dt = dt.date(2021,1,1)
end_dt = dt.date(2025,2,1)

- combine all dividends data

In [9]:
# dates = pd.date_range(start_dt, end_dt).date
# td_l = []
# for d in dates:
# 	date_s = d.strftime("%Y-%m-%d")
# 	tdf = pd.DataFrame()
# 	try:
# 		tdf = pd.read_csv(f'data_by_date/dividends/{date_s}.csv')
# 	except Exception as e:
# 		qt.log.warning(f"file not found for date : {d}")
# 	qt.log.info(f"[{d}] tdf.shape = {tdf.shape}")
# 	td_l.append(tdf)

# td = pd.concat(td_l).sort_values(['Date', 'Code']).reset_index(drop=True)
# td.to_csv('data_by_date/all_dividends_till_20260201.csv', index=False)

- one symbol only

In [10]:
symbol = "GOOGL"
ts = get_data_by_symbol_filename(symbol=symbol, filename='split')
td = get_data_by_symbol_filename(symbol=symbol, filename='div')
ti = get_data_by_symbol_filename(symbol=symbol, filename='intraday')

[JUSTY.LOG]	2026-02-17 23:58:41,190 - qt.common.help - INFO - df.shape = (1, 2)
[JUSTY.LOG]	2026-02-17 23:58:41,192 - qt.common.help - INFO - df.shape = (7, 8)
[JUSTY.LOG]	2026-02-17 23:58:41,393 - qt.common.help - INFO - df.shape = (921908, 7)


In [11]:
all_d = pd.read_csv('data_by_date/all_dividends_till_20260201.csv')

In [12]:
td
all_d[all_d['Code'] == symbol]
ts

,date,declarationDate,recordDate,paymentDate,period,value,unadjustedValue,currency
0,2024-06-10,2024-04-25,2024-06-10,2024-06-17,Quarterly,0.20,0.20,USD
1,2024-09-09,2024-07-23,2024-09-09,2024-09-16,Quarterly,0.20,0.20,USD
2,2024-12-09,2024-10-28,2024-12-09,2024-12-16,Quarterly,0.20,0.20,USD
3,2025-03-10,2025-02-04,2025-03-10,2025-03-17,Quarterly,0.20,0.20,USD
4,2025-06-09,2025-04-23,2025-06-09,2025-06-16,Quarterly,0.21,0.21,USD
5,2025-09-08,2025-07-21,2025-09-08,2025-09-15,Quarterly,0.21,0.21,USD
6,2025-12-08,2025-10-21,2025-12-08,2025-12-15,Quarterly,0.21,0.21,USD


,Code,Ex,Date,Dividend,Currency
610361,GOOGL,US,2024-06-10,0.20,USD
649754,GOOGL,US,2024-09-09,0.20,USD
688040,GOOGL,US,2024-12-09,0.20,USD


,date,split
0,2022-07-18,20.000000/1.000000


In [ ]:
ti = add_dt_us_intraday(ti)
ti = ti.sort_values('datetime_us').reset_index(drop=True)
ti['close'] = ti['close'].ffill()

In [ ]:
til = pd.DataFrame()

In [ ]:
ti['time'].min()
ti['time'].max()

In [ ]:
ti[ti['time'] == dt.time(9,30)].shape
ti[ti['time'] == dt.time(16,0)].shape
ti['date'].nunique()

In [ ]:
ti[ (ti['time'] <= dt.time(16,0))]

In [ ]:
start_date_sym = ti['date'].min()
end_date_sym = ti['date'].max()
tl = pd.date_range(start_date_sym, end_date_sym).date
tl = pd.DataFrame({'date' : tl})

In [ ]:
tl

In [ ]:
ti[ti['date'] == dt.date(2020, 12, 31)]

In [ ]:
def minutes_since_midnight(t):
	return t.hour * 60 + t.minute + t.second / 60


tis = ti.groupby('date').agg(
	min_time=("time", "min"),
	max_time=("time", "max"),
	count_time=("time", "count"),
	count_valid_close=("close", lambda x: (~x.isna()).sum()),
	count_before_open=("time", lambda x: (x<dt.time(9,30)).sum()),
	count_after_close=("time", lambda x: (x>dt.time(16,0)).sum()),
)

OPEN_MIN = 9 * 60 + 30
CLOSE_MIN = 16 * 60 + 0
tis["minutes_early_from_open"] = (OPEN_MIN-tis["min_time"].apply(minutes_since_midnight)).clip(lower=0)
tis["minutes_late_after_close"] = (tis["max_time"].apply(minutes_since_midnight) - CLOSE_MIN).clip(lower=0)
qt.view(tis)

In [ ]:
tis.sort_values('date') [['minutes_early_from_open', 'minutes_late_after_close']].plot()

In [ ]:
ti[(ti['time'] >= dt.time(9,30)) & (ti['time'] <= dt.time(16,0)) & (ti['date'] == dt.date(2024, 12, 11))]

In [ ]:
start_date_sym = ti['date'].min()
end_date_sym = ti['date'].max()
pd.date_range(start_date_sym, end_date_sym)

In [ ]:
# ti.set_index('datetime_us')['close'].plot()